In [ ]:
# ── MLOps bootstrap (auto-injected by inject_mlops_cell.py) ──────────────────
import os, warnings, mlflow
warnings.filterwarnings("ignore")

SEED = 42
import random, numpy as np
random.seed(SEED)
np.random.seed(SEED)
try:
    import torch; torch.manual_seed(SEED)
except ImportError:
    pass
try:
    import tensorflow as tf; tf.random.set_seed(SEED)
except ImportError:
    pass

_nb_name = os.path.basename(os.path.abspath("__file__") if "__file__" in dir() else ".").replace(".ipynb","")
mlflow.set_tracking_uri("sqlite:///" + str(Path(__file__).parent.parent.parent / "mlflow.db")
                        if "__file__" in dir() else "sqlite:///mlflow.db")
_exp = mlflow.set_experiment(_nb_name or "unnamed_notebook")
print(f"MLflow experiment: {_exp.name}")


# 93 — Planning + Execution Agent

## Goal
Explicitly **separate** a **Planner** node from **Worker** nodes,
with shared memory. The planner breaks a task into steps,
the worker executes each, and the planner checks progress.

## What You'll Learn
| Concept | Detail |
|---------|--------|
| **Plan-then-execute** | Two-phase agent architecture |
| **Shared state** | Both nodes read/write to a common TypedDict |
| **Step tracking** | Planner generates steps, worker executes one at a time |
| **Supervision loop** | Planner reviews after each step |

## Stack
- `langgraph` — StateGraph with planner and worker nodes
- `langchain-ollama` — local LLM
- Ollama `qwen3.5:9b`

In [ ]:
import os, warnings, json
os.environ["USE_TF"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore")

from typing import TypedDict
from langgraph.graph import StateGraph, END
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage

llm = ChatOllama(model="qwen3.5:9b", temperature=0)
print("All imports OK")

## Step 1 — Worker Tools (Simulated)

The worker has access to simulated tools for data analysis.

In [ ]:
# Simulated tools the worker can use
TOOL_REGISTRY = {
    "fetch_sales_data": lambda: {"Q1": 1_200_000, "Q2": 1_450_000, "Q3": 1_380_000, "Q4": 1_620_000},
    "fetch_customer_data": lambda: {"total": 4500, "new_q4": 320, "churned_q4": 85, "nps": 62},
    "fetch_expense_data": lambda: {"salaries": 2_100_000, "marketing": 450_000, "ops": 680_000, "r_and_d": 920_000},
    "calculate_metrics": lambda data: {
        "total_revenue": sum(data.values()) if isinstance(data, dict) else 0,
        "avg_quarterly": sum(data.values()) / len(data) if data else 0
    },
    "generate_summary": lambda findings: f"Analysis complete. Key findings: {json.dumps(findings)}"
}

def execute_tool(tool_name: str, arg=None):
    """Execute a named tool from the registry."""
    if tool_name not in TOOL_REGISTRY:
        return f"ERROR: Tool '{tool_name}' not found. Available: {list(TOOL_REGISTRY.keys())}"
    fn = TOOL_REGISTRY[tool_name]
    try:
        return fn(arg) if arg else fn()
    except TypeError:
        return fn()

print(f"Available tools: {list(TOOL_REGISTRY.keys())}")

## Step 2 — State and Node Definitions

```
planner → worker → supervisor → [done?]
                                  → NO:  worker (next step)
                                  → YES: finalize → END
```

In [ ]:
class PlanExecState(TypedDict):
    task: str
    plan: list          # List of step dicts: {"step": str, "tool": str}
    current_step: int
    step_results: list  # Results from each completed step
    memory: dict        # Shared memory between nodes
    final_output: str

print("State schema defined")

In [ ]:
def planner_node(state: PlanExecState) -> PlanExecState:
    """Generate a step-by-step plan for the task."""
    print(f"\n🗺️ PLANNER: Creating plan for: {state['task'][:80]}")
    
    tools_desc = ", ".join(TOOL_REGISTRY.keys())
    msg = llm.invoke([
        SystemMessage(content=f"""You are a task planner. Break the task into 3-5 steps.
Available tools: {tools_desc}

Return a JSON array of steps:
[{{"step": "description", "tool": "tool_name"}}]

Each step should use one tool. Return ONLY the JSON array."""),
        HumanMessage(content=state["task"])
    ])
    
    raw = msg.content
    if "<think>" in raw:
        raw = raw.split("</think>")[-1].strip()
    if "```" in raw:
        raw = raw.split("```")[1]
        if raw.startswith("json"):
            raw = raw[4:]
        raw = raw.strip()
    
    try:
        start = raw.find("[")
        end = raw.rfind("]") + 1
        plan = json.loads(raw[start:end])
    except (json.JSONDecodeError, ValueError):
        plan = [
            {"step": "Fetch sales data", "tool": "fetch_sales_data"},
            {"step": "Fetch customer data", "tool": "fetch_customer_data"},
            {"step": "Fetch expense data", "tool": "fetch_expense_data"},
            {"step": "Generate summary", "tool": "generate_summary"}
        ]
    
    for i, s in enumerate(plan):
        print(f"   Step {i+1}: {s['step']} → [{s['tool']}]")
    
    return {**state, "plan": plan, "current_step": 0, "step_results": []}

print("Planner node defined")

In [ ]:
def worker_node(state: PlanExecState) -> PlanExecState:
    """Execute the current step using the specified tool."""
    idx = state["current_step"]
    step = state["plan"][idx]
    print(f"\n⚙️ WORKER: Step {idx+1}/{len(state['plan'])} — {step['step']}")
    
    tool_name = step.get("tool", "")
    
    # Pass accumulated data to tools that need it
    if tool_name == "calculate_metrics" and state["memory"].get("sales_data"):
        result = execute_tool(tool_name, state["memory"]["sales_data"])
    elif tool_name == "generate_summary" and state["step_results"]:
        result = execute_tool(tool_name, state["step_results"])
    else:
        result = execute_tool(tool_name)
    
    print(f"   Result: {json.dumps(result, default=str)[:150]}")
    
    # Update memory with results
    memory = dict(state["memory"])
    if "sales" in tool_name:
        memory["sales_data"] = result
    elif "customer" in tool_name:
        memory["customer_data"] = result
    elif "expense" in tool_name:
        memory["expense_data"] = result
    
    step_results = list(state["step_results"]) + [
        {"step": step["step"], "tool": tool_name, "result": result}
    ]
    
    return {**state, "step_results": step_results, "memory": memory,
            "current_step": idx + 1}

print("Worker node defined")

In [ ]:
def supervisor_node(state: PlanExecState) -> PlanExecState:
    """Check if all steps are done."""
    done = state["current_step"] >= len(state["plan"])
    remaining = len(state["plan"]) - state["current_step"]
    print(f"\n📋 SUPERVISOR: {state['current_step']}/{len(state['plan'])} steps done. "
          f"{'All complete!' if done else f'{remaining} remaining'}")
    return state

def supervisor_route(state: PlanExecState) -> str:
    if state["current_step"] >= len(state["plan"]):
        return "finalize"
    return "worker"

def finalize_node(state: PlanExecState) -> PlanExecState:
    """Generate final output from all step results."""
    print("\n✅ FINALIZE: Generating final report...")
    
    results_text = json.dumps(state["step_results"], indent=2, default=str)
    msg = llm.invoke([
        SystemMessage(content="""Synthesize these step results into a clear executive summary.
Include key numbers and insights. 150-250 words. No thinking tags."""),
        HumanMessage(content=f"Task: {state['task']}\n\nStep Results:\n{results_text}")
    ])
    
    output = msg.content
    if "<think>" in output:
        output = output.split("</think>")[-1].strip()
    
    return {**state, "final_output": output}

print("Supervisor and finalize nodes defined")

In [ ]:
# Build the graph
graph = StateGraph(PlanExecState)

graph.add_node("planner", planner_node)
graph.add_node("worker", worker_node)
graph.add_node("supervisor", supervisor_node)
graph.add_node("finalize", finalize_node)

graph.set_entry_point("planner")
graph.add_edge("planner", "worker")
graph.add_edge("worker", "supervisor")
graph.add_conditional_edges("supervisor", supervisor_route, {
    "worker": "worker",
    "finalize": "finalize"
})
graph.add_edge("finalize", END)

plan_exec_app = graph.compile()
print("Plan-and-execute graph compiled!")

In [ ]:
# Run the planner + executor
result = plan_exec_app.invoke({
    "task": "Analyze the company's annual performance: gather sales, customer, and expense data, then provide an executive summary.",
    "plan": [],
    "current_step": 0,
    "step_results": [],
    "memory": {},
    "final_output": ""
})

print("\n" + "=" * 60)
print("EXECUTIVE SUMMARY")
print("=" * 60)
print(result["final_output"])

print(f"\nPlan had {len(result['plan'])} steps, {len(result['step_results'])} executed")

## 🧠 Key Concepts Recap

| Concept | Implementation |
|---------|---------------|
| **Planner** | LLM generates a step-by-step plan as JSON |
| **Worker** | Executes one step at a time using tools |
| **Supervisor** | Checks progress, routes to next step or finalize |
| **Shared memory** | `memory` dict accumulates results across steps |
| **Step tracking** | `current_step` counter tracks progress |

## Architecture
```
Planner → Worker → Supervisor ─┐
           ↑                    │
           │    more steps?     │
           │       YES ─────────┘
           │       NO
           │       ↓
           └── Finalize → END
```

## Extending This Project
- Dynamic replanning if a tool fails
- Parallel step execution for independent tasks
- Human approval gate before critical steps
- Plan visualization with Mermaid diagrams